# 4-практика · Аномалияларды издөө (Anomaly Detection)

**Кыргыз Республикасынын Эсептөө палатасы · AI тренинги · 2-күн**

Кечээ моделге **туура жоопторду** бердик: «бул мекеме — Жогорку тобокелдик, бул — Төмөн».
Бүгүн жооптор **ЖОК**. Машина өзү издейт — бул **көзөмөлсүз үйрөнүү (Unsupervised Learning)**.

**Кантип иштетүү керек (кечээгидей эле):**
- Код блогун тандап, ▶ баскычын басыңыз (же **Shift + Enter**)
- Блокторду **иретте, жогорудан ылдый** иштетиңиз

⚠️ **Эч нерсени бузуп алуу мүмкүн эмес.** Эң жаманы — баарын өчүрүп, кайра баштайбыз 🙂

## 0-кадам · Даярдык

Кечээгидей эле — куралдарды орнотобуз. Бир мүнөтчө күтө туруңуз.

In [ ]:
%pip install -q pandas scikit-learn matplotlib

## 1-кадам · Кечээги маалымат — бирок жообу ЖОК

Кечээги 250 мекемени эстейсизби? Ошол эле маалыматты түзөбүз, бирок бир айырма менен:

**`тобокелдик` мамычасы жок.** Эч ким бизге «бул кооптуу, бул таза» деп айтпайт.

⚠️ Маалымат мурдагыдай эле толугу менен **синтетикалык** (ойдон чыгарылган).

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(12)   # seed — жыйынтык ар дайым бирдей болушу үчүн
n = 250

мекемелер = pd.DataFrame({
    "аймак": rng.choice(["Бишкек","Ош","Чүй","Ысык-Көл","Жалал-Абад","Нарын","Талас","Баткен"], n),
    "тип": rng.choice(["мектеп","оорукана","муниципалдык ишкана","мамлекеттик мекеме","айыл өкмөтү"], n),
    "бюджет_млн_сом": np.round(rng.uniform(5, 500, n), 1),
    "мурунку_бузуулар": rng.poisson(1.4, n).clip(0, 8),
    "бир_булак_пайыз": np.round(rng.beta(2, 4, n) * 90, 0),
    "кечигүү_күн": rng.poisson(8, n).clip(0, 60),
    "четтөө_пайыз": np.round(np.abs(rng.normal(8, 7, n)).clip(0, 30), 1),
})

print("Даяр! Мекемелердин саны:", len(мекемелер))
print("Мамычалар:", list(мекемелер.columns))
print()
print("Байкадыңызбы? «тобокелдик» мамычасы ЖОК — жооптор жок!")

## 2-кадам · Кластерлөө (Clustering): машина өзү топторго бөлөт

Суроо: бул 250 мекеме табигый түрдө кандай **топторго** бөлүнөт?

**K-Means** алгоритмине айтабыз: «3 топ тап». Ал эмне боюнча бөлүштү — **өзү чечет**.
Биз эки белгини карайлы: бир булактан сатып алуулар (%) жана бюджеттен четтөө (%).

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

белгилер_2 = мекемелер[["бир_булак_пайыз", "четтөө_пайыз"]]

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
мекемелер["топ"] = kmeans.fit_predict(белгилер_2)

түстөр = ["#2C5F2D", "#C9A227", "#5F6F66"]
plt.figure(figsize=(9, 6))
for т in range(3):
    б = мекемелер[мекемелер["топ"] == т]
    plt.scatter(б["бир_булак_пайыз"], б["четтөө_пайыз"],
                color=түстөр[т], label=f"{т+1}-топ ({len(б)} мекеме)", alpha=0.7)
plt.xlabel("Бир булактан сатып алуулар (%)")
plt.ylabel("Бюджеттен четтөө (%)")
plt.title("K-Means: машина тапкан 3 топ (эч ким үйрөткөн жок!)")
plt.legend()
plt.show()

**Талкууга суроо:** Ар бир түскө карап көрүңүз. Машина топторду эмненин негизинде бөлдү?
Кайсы топ аудитордун көңүлүн биринчи бурушу керек?

Байкаңыз: биз машинага «жакшы/жаман» деп айтпадык. Ал жөн гана **окшошту окшош менен** топтоду.
Атын коюу — **адамдын** иши.

## 3-кадам · Жаңы маселе: төлөмдөрдүн ичинен «башкача» бирөөнү табуу

Эми чыныгы аудитордук маселе. Бир министрликтин **400 төлөмүн** алабыз:

| Мамыча | Мааниси |
|---|---|
| `берүүчү` | ким менен келишим түзүлгөн |
| `категория` | эмне сатылып алынган |
| `сумма_миң_сом` | төлөмдүн өлчөмү (миң сом) |
| `ай` | кайсы айда төлөнгөн (1–12) |
| `жума_күнү` | кайсы күнү (1=дүйшөмбү ... 7=жекшемби) |

Алардын арасында **бир нече кадимки эмес төлөм катылган**. Кайсылары экенин билбейбиз — машина тапсын!

In [ ]:
rng2 = np.random.default_rng(7)
m = 400

категориялар = ["канцелярия", "оңдоо иштери", "транспорт", "эмерек", "техника"]
орточо_сумма = {"канцелярия": 45, "оңдоо иштери": 380, "транспорт": 210, "эмерек": 120, "техника": 260}

төлөмдөр = pd.DataFrame({
    "берүүчү": rng2.choice(["Ак-Марал ЖЧК","Бек-Курулуш","Нур-Сервис","Алтын-Т","Эмгек ОсОО","Багыт ЖЧК"], m),
    "категория": rng2.choice(категориялар, m),
    "ай": rng2.integers(1, 13, m),
    "жума_күнү": rng2.choice([1,2,3,4,5], m),          # кадимки төлөмдөр — жумуш күндөрү
})
төлөмдөр["сумма_миң_сом"] = [
    round(орточо_сумма[к] * rng2.lognormal(0, 0.35), 1) for к in төлөмдөр["категория"]]

# === Катылган аномалиялар (кийин салыштыруу үчүн белгилеп коёбуз — модель муну КӨРБӨЙТ!) ===
төлөмдөр["катылган"] = False
аномалиялар = pd.DataFrame({
    "берүүчү":   ["Алтын-Т","Алтын-Т","Багыт ЖЧК","Нур-Сервис","Бек-Курулуш","Ак-Марал ЖЧК","Алтын-Т","Эмгек ОсОО"],
    "категория": ["техника","техника","оңдоо иштери","канцелярия","оңдоо иштери","эмерек","транспорт","техника"],
    "ай":        [12, 12, 12, 6, 3, 9, 12, 7],       # акыркысы — атайын «жашырынган» 🙂
    "жума_күнү": [6, 7, 6, 7, 5, 6, 7, 2],           # көбү — дем алыш күндөрү!
    "сумма_миң_сом": [2450.0, 1980.0, 3100.0, 890.0, 2700.0, 1150.0, 1890.0, 470.0],
    "катылган": True,
})
төлөмдөр = pd.concat([төлөмдөр, аномалиялар], ignore_index=True)
төлөмдөр = төлөмдөр.sample(frac=1, random_state=3).reset_index(drop=True)   # аралаштырабыз

print("Даяр! Төлөмдөрдүн саны:", len(төлөмдөр))
төлөмдөр.drop(columns="катылган").head(8)

408 төлөм. Көз менен карап, аномалияны табуу кыйын. Адегенде сүрөткө салып көрөлү:

In [ ]:
plt.figure(figsize=(10, 5.5))
plt.scatter(төлөмдөр["ай"] + rng2.uniform(-0.25, 0.25, len(төлөмдөр)),
            төлөмдөр["сумма_миң_сом"], alpha=0.55, color="#2C5F2D")
plt.xlabel("Ай (1 = январь ... 12 = декабрь)")
plt.ylabel("Сумма (миң сом)")
plt.title("408 төлөм: ай жана сумма")
plt.show()

print("Кээ бир чекиттер «үйүрдөн» алыс турганын байкадыңызбы?")

## 4-кадам · Isolation Forest: «жалгыздарды» табуучу алгоритм

**Идеясы жөнөкөй:** маалыматты туш келди суроолор менен бөлө беребиз.
Кадимки чекитти «жалгыздатуу» үчүн көп суроо керек — ал үйүрдүн ичинде.
Аномалияны **аз эле суроо** жалгыздатат — ал четте турат.

Ошондуктан аты — **Isolation** (жалгыздатуу) **Forest** (токой — көп дарак).

Моделге 3 белгини беребиз: **сумма, ай, жума күнү**. Жооп бербейбиз — өзү издейт!

In [ ]:
from sklearn.ensemble import IsolationForest

белгилер = төлөмдөр[["сумма_миң_сом", "ай", "жума_күнү"]]

модель = IsolationForest(contamination=0.03, random_state=42)
модель.fit(белгилер)

# Ар бир төлөмгө «аномалдуулук баллы»: канчалык төмөн болсо — ошончолук кадимки эмес
төлөмдөр["балл"] = модель.decision_function(белгилер)
төлөмдөр["шектүү"] = модель.predict(белгилер) == -1

print(f"Модель {төлөмдөр['шектүү'].sum()} төлөмдү «кадимки эмес» деп белгиледи (408дин ичинен)")

Кайсыларын белгилегенин сүрөттөн көрөлү — алтын түстүү чекиттер:

In [ ]:
plt.figure(figsize=(10, 5.5))
кадимки = төлөмдөр[~төлөмдөр["шектүү"]]
шектүү = төлөмдөр[төлөмдөр["шектүү"]]
plt.scatter(кадимки["ай"] + rng2.uniform(-0.25, 0.25, len(кадимки)),
            кадимки["сумма_миң_сом"], alpha=0.4, color="#2C5F2D", label="кадимки")
plt.scatter(шектүү["ай"], шектүү["сумма_миң_сом"],
            color="#C9A227", s=110, edgecolor="#173F31", linewidth=1.5, label="ШЕКТҮҮ", zorder=5)
plt.xlabel("Ай")
plt.ylabel("Сумма (миң сом)")
plt.title("Isolation Forest тапкан аномалиялар")
plt.legend()
plt.show()

## 5-кадам · Аудитордун тизмеси: эң шектүү 10 төлөм

Баллы боюнча иреттеп, эң «кадимки эмес» 10 төлөмдү карайлы.
**Суроо менен окуңуз:** нээ үчүн машина дал ушуларды белгиледи?

In [ ]:
тизме = төлөмдөр.sort_values("балл").head(10)[
    ["берүүчү", "категория", "ай", "жума_күнү", "сумма_миң_сом", "балл"]]

күндөр = {1:"дүйшөмбү",2:"шейшемби",3:"шаршемби",4:"бейшемби",5:"жума",6:"ИШЕМБИ ⚠️",7:"ЖЕКШЕМБИ ⚠️"}
тизме = тизме.copy()
тизме["жума_күнү"] = тизме["жума_күнү"].map(күндөр)
тизме["балл"] = тизме["балл"].round(3)
тизме

**Талкууга суроо (2 мүнөт):** Тизмедеги төлөмдөрдүн жалпылыгы эмнеде?

- Чоң суммалар?
- Дем алыш күндөрү төлөнгөндөр?
- Декабрь айы? (жыл акырында бюджетти «өздөштүрүү» 🙂)

Аудитор катары кайсынысын биринчи текшермексиз?

## 6-кадам · Текшерүү: машина катылгандарды таптыбы?

Биз 8 аномалияны атайын каткан элек. Модель канчасын тапты экен?

In [ ]:
табылды = (төлөмдөр["катылган"] & төлөмдөр["шектүү"]).sum()
жалпы = төлөмдөр["катылган"].sum()

print(f"Катылган аномалиялар:  {жалпы}")
print(f"Модель тапканы:        {табылды} ({табылды/жалпы:.0%})")
print()
print("Кечээги сабакты эстейли: Recall =", f"{табылды/жалпы:.0%}",
      "— бардык «кооптуулардын» канчасын кармадык.")
табылбады = төлөмдөр[төлөмдөр["катылган"] & ~төлөмдөр["шектүү"]]
if len(табылбады):
    print(f"\nТабылбай калганы ({len(табылбады)}):")
    print(табылбады[["берүүчү","категория","ай","жума_күнү","сумма_миң_сом"]].to_string(index=False))

Байкаңыз: **баары табылбашы мүмкүн** — «үйүргө» жакын жашырынган аномалия машинага да кыйын.
Мисалы, келишимди майдалап бөлүү (splitting) — ар бир төлөм өзүнчө «кадимки» көрүнөт.
Аны табуу үчүн башка белгилер керек (берүүчү боюнча топтоо ж.б.) — бул курал канчалык
**белгилерге (features)** көз каранды экенин дагы бир жолу көрсөтөт.

## 7-кадам · 🖊 СИЗДИН ЭКСПЕРИМЕНТ: «канча пайызы шектүү?» деген жөндөө

`contamination = 0.03` — «болжол менен 3% шектүү болушу мүмкүн» деген биздин божомолубуз.
Бул кечээги **босого (threshold)** сыяктуу эле — аны **БИЗ** тандайбыз.

**Тапшырма:** төмөндө `0.03`тү адегенде `0.01`ге, анан `0.08`ге өзгөртүп, блокту кайра иштетиңиз.
Тизме кандай өзгөрөт? Кайсы жөндөө аудиторго ыңгайлуу?

In [ ]:
жөндөө = 0.03      # 🖊 БУЛ САНДЫ ӨЗГӨРТҮҢҮЗ: 0.03 → 0.01 (андан кийин 0.08 менен да ойноп көрүңүз)

модель2 = IsolationForest(contamination=жөндөө, random_state=42)
модель2.fit(белгилер)
шектүү2 = модель2.predict(белгилер) == -1

табылды2 = (төлөмдөр["катылган"] & шектүү2).sum()
print(f"Жөндөө (contamination) = {жөндөө}")
print(f"  Текшерүүгө сунушталган төлөмдөр: {шектүү2.sum()}")
print(f"  Катылган 8 аномалиядан табылганы: {табылды2}")
print()
print("Аз текшеребиз → айрымы качып кетет. Көп текшеребиз → ресурс короойт.")
print("Тааныш танданубу? Кечээги FN менен FP! 🙂")

## 8-кадам · Корутунду

✓ **Кластерлөө** — машина окшошту окшош менен топтойт; топтун атын адам коёт

✓ **Аномалия детекциясы** — «үйүрдөн» четтегендерди табат; жооптор такыр керек эмес

✓ **Аномалия ≠ бузуу.** Бул — далил эмес, **текшерүүгө багыт**. Чечим ар дайым аудитордуку

✓ Жөндөөлөр (канча пайыз, кайсы белгилер) — кечээгидей эле **адамдын чечими**

**Тыныгуудан кийин:** машиналар сүрөттү кантип «көрөт» — Computer Vision жана тирүү демонстрациялар 📷